In [21]:
import numpy as np
import random
from sklearn.linear_model import LinearRegression
from scipy import stats
import json

In [7]:
np.random.seed(2026)

Генрация выборки n = 50
$$
(\xi_1, \xi_2, \xi_3, \xi_4, \xi_5, \eta)
$$
$$
\xi_k \sim R(-1,1)
$$

$$
\eta \sim N(2 + 3x_1 - 2x_2 + x_3 + x_4 - x_5, 1.5^2)
$$

$$
x_k - \text{значение, которое принимает } \xi_k
$$


In [8]:
import numpy as np

n = 50

ksi = np.random.uniform(-1, 1, (n, 5))

mean = 2 + 3*ksi[:,0] - 2*ksi[:,1] + ksi[:,2] + ksi[:,3] - ksi[:,4]

eta = np.random.normal(mean, 1.5)

vec = np.concatenate([ksi, eta.reshape(-1, 1)], axis=1)

In [13]:
with open('data.json', 'w', encoding='utf-8') as f:
    json.dump(vec.tolist(), f, indent=2)

In [11]:
print(vec.shape)
print(vec[:10])

(50, 6)
[[-0.56130873 -0.17397653  0.95327096 -0.82220196 -0.04142028 -0.50695418]
 [ 0.97510099 -0.60371782  0.82343036  0.11650023  0.56645894  5.57474782]
 [-0.39504682 -0.20539801 -0.40806537  0.78817692 -0.90364979  4.15590303]
 [-0.10986471  0.12378778 -0.27257074  0.38898616  0.84673616  1.94088806]
 [-0.93881386 -0.75718912 -0.53618726  0.2857939  -0.5281328   0.29307876]
 [ 0.93373154 -0.70166573 -0.90620712 -0.05909771  0.36515497  6.2303456 ]
 [ 0.75876632  0.36584074  0.04057325 -0.06598888 -0.4851988   5.96377173]
 [ 0.01605656  0.14893959 -0.01971623 -0.58564    -0.09823288  0.81056611]
 [ 0.87985772 -0.37649255 -0.95399486 -0.50180832 -0.57803312  6.3141511 ]
 [-0.59474656  0.34199468  0.48449089  0.7091047  -0.83130243  0.97336732]]


$$
\xi_1 предск = bo + b2*\xi2 + b3*\xi3 + b4*\xi4 + b5*\xi5
$$

In [ ]:

n = 50

bad_variables = []
all_results = []

for i in range(5):
    remaining_indices = [j for j in range(5) if j != i and j not in bad_variables]
    
    if len(remaining_indices) == 0:
        print(f"Нет доступных факторов для ξ{i+1} (все остальные уже исключены)")
        all_results.append({
            'variable': f'ξ{i+1}',
            'R²': None,
            'status': 'не проверена (нет факторов)',
            'excluded': False
        })
        continue
    
    y = vec[:, i]
    
    X_factors = vec[:, remaining_indices]
    
    factor_names = [f'ξ{j+1}' for j in remaining_indices]
    
    
    model = LinearRegression()
    model.fit(X_factors, y)
    y_pred = model.predict(X_factors)
    
    RSS = np.sum((y - y_pred)**2)
    TSS = np.sum((y - np.mean(y))**2)
    R2 = 1 - RSS/TSS
    
    print(f"  RSS = {RSS:.4f}")
    print(f"  TSS = {TSS:.4f}")
    print(f"  R² = {R2:.4f}")
    
    if R2 > 0.7:
        bad_variables.append(i)
        excluded = True
    else:
        status = "ОСТАВЛЕНА"
        excluded = False
    
    all_results.append({
        'variable': f'ξ{i+1}',
        'R²': R2,
        'status': status,
        'excluded': excluded,
        'factors': factor_names
    })


for res in all_results:
    if res['R²'] is not None:
        print(f"  {res['variable']}: R² = {res['R²']:.4f} → {res['status']}")
    else:
        print(f"  {res['variable']}: {res['status']}")

print(f"\nИсключенные переменные: {[f'ξ{b+1}' for b in bad_variables]}")
print(f"Оставленные переменные: {[f'ξ{i+1}' for i in range(5) if i not in bad_variables]}")

if len(bad_variables) == 0:
    print("\nМультиколлинеарности нет. Все переменные независимы.")
else:
    print(f"\nРекомендуется удалить {[f'ξ{b+1}' for b in bad_variables]}")

results_to_save = {
    'bad_variables': [f'ξ{b+1}' for b in bad_variables],
    'good_variables': [f'ξ{i+1}' for i in range(5) if i not in bad_variables],
    'details': all_results,
    'n_samples': n,
    'threshold': 0.7
}



  RSS = 15.2112
  TSS = 16.6166
  R² = 0.0846
  RSS = 15.2170
  TSS = 16.5655
  R² = 0.0814
  RSS = 17.0167
  TSS = 17.6657
  R² = 0.0367
  RSS = 17.7736
  TSS = 19.0721
  R² = 0.0681
  RSS = 16.5295
  TSS = 18.6541
  R² = 0.1139
  ξ1: R² = 0.0846 → ОСТАВЛЕНА
  ξ2: R² = 0.0814 → ОСТАВЛЕНА
  ξ3: R² = 0.0367 → ОСТАВЛЕНА
  ξ4: R² = 0.0681 → ОСТАВЛЕНА
  ξ5: R² = 0.1139 → ОСТАВЛЕНА

Исключенные переменные: []
Оставленные переменные: ['ξ1', 'ξ2', 'ξ3', 'ξ4', 'ξ5']

Мультиколлинеарности нет. Все переменные независимы.


In [ ]:
with open('1a.json', 'w', encoding='utf-8') as f:
    json.dump(results_to_save, f, indent=2, ensure_ascii=False)


$\eta = \beta_0 + \sum_{k=1}^{5} \beta_k \xi_k$ 
Определиь уравнение регрессии и проверить значимость коэффициентов

$$\mathbf{F} = \mathbf{X}^T\mathbf{X}$$
$$\mathbf{e} = \mathbf{Y} - \mathbf{X}\hat{\boldsymbol{\beta}}$$

$$t_i = \frac{\hat{\beta}_i - \beta_i}{\sqrt{RSS \cdot (\mathbf{X}^T\mathbf{X})^{-1}_{ii}}}\sqrt{n-p} \sim t(n-p)$$
$$\text{H0: }  \beta_i = 0$$

In [ ]:
Y = vec[:, 5]
X = vec[:, 0:5]

model = LinearRegression(fit_intercept=True)
model.fit(X, Y)

beta_hat = np.append(model.intercept_, model.coef_)

Y_pred = model.predict(X)

residuals = Y - Y_pred
RSS = np.sum(residuals**2)

n = len(Y)
p = X.shape[1] + 1  # 6 параметров

# Для стандартных ошибок нужна матрица X с единицами
X_with_intercept = np.column_stack([np.ones(n), X])

sigma2_hat = RSS / (n - p)

# Обратная матрица (X'X)^(-1)
X_inv = np.linalg.inv(X_with_intercept.T @ X_with_intercept)

# Стандартные ошибки
SE = np.sqrt(sigma2_hat * np.diag(X_inv))

t_stats = beta_hat / SE

df = n - p

p_values = 2 * stats.t.sf(np.abs(t_stats), df=df)

print(f"Степени свободы: df = {df}")
print(f"σ̂² = {sigma2_hat:.4f}")

print(f"\n{'Коэфф.':<8} {'β̂':<10} {'SE':<10} {'t-стат':<10} {'p-value':<10} {'Значим':<8}")
print("-"*70)

for i, (b, se, t, p) in enumerate(zip(beta_hat, SE, t_stats, p_values)):
    significant = "ДА" if p < 0.05 else "нет"
    print(f"β{i:<7} {b:<10.4f} {se:<10.4f} {t:<10.4f} {p:<10.4f} {significant:<8}")

alpha = 0.05
t_critical = stats.t.ppf(1 - alpha/2, df=df)
print(f"\nКритическое значение t({df}) = ±{t_critical:.3f}")

6
С использованием sklearn:
Степени свободы: df = 44
σ̂² = 2.4748

Коэфф.   β̂         SE         t-стат     p-value    Значим  
----------------------------------------------------------------------
β0       1.9699     0.2259     8.7205     0.0000     ДА      
β1       2.9485     0.4034     7.3098     0.0000     ДА      
β2       -2.1777    0.4033     -5.4000    0.0000     ДА      
β3       0.5454     0.3814     1.4301     0.1597     нет     
β4       0.3370     0.3732     0.9032     0.3713     нет     
β5       -1.0800    0.3869     -2.7911    0.0077     ДА      

Критическое значение t(44) = ±2.015


In [26]:
results_compact = {
    "model": "η = β₀ + Σ βₖξₖ",
    "n": n,
    "df": df,
    "sigma2": round(sigma2_hat, 4),
    "R2": round(model.score(X, Y), 4),
    "coefficients": {
        f"β{i}": {
            "estimate": round(beta_hat[i], 4),
            "se": round(SE[i], 4),
            "t_stat": round(t_stats[i], 4),
            "p_value": round(p_values[i], 4),
            "significant": round(p_values[i], 4)
        }
        for i in range(len(beta_hat))
    }
}

with open('1b.json', 'w', encoding='utf-8') as f:
    json.dump(results_compact, f, indent=2, ensure_ascii=False)

In [ ]:
from sklearn.linear_model import LinearRegression

Y = vec[:, 5]
X = vec[:, 0:5]

model = LinearRegression(fit_intercept=True)
model.fit(X, Y)
y_pred = model.predict(X)

RSS = np.sum((Y - y_pred)**2)
TSS = np.sum((Y - np.mean(Y))**2)
R2 = 1 - RSS/TSS

print(f"RSS = {RSS:.4f}")
print(f"TSS = {TSS:.4f}")
print(f"R² = {R2:.4f}")

RSS = 108.8929
TSS = 302.8171
R² = 0.6404


In [32]:
result = {"R^2": R2}

with open('1c.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

$$
\eta = \beta_0
$$

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from scipy import stats
import matplotlib.pyplot as plt
import json

# ξ = 0

Y = vec[:, 5]
X = vec[:, 0:5]
n = len(Y)

# 1. Исходная модель
model = LinearRegression()
model.fit(X, Y)

beta_hat = np.append(model.intercept_, model.coef_)
y_pred_original = model.predict(X)
residuals = Y - y_pred_original
sigma_hat = np.std(residuals, ddof=1)

x_new = np.zeros((1, 5))  # [0, 0, 0, 0, 0]

# Прогноз в этой точке — это просто β₀
y0_point = model.intercept_

print(f"Модель: η = {model.intercept_:.4f}", end="")
for i, coef in enumerate(model.coef_):
    print(f" + ({coef:.4f})·ξ{i+1}", end="")
print()
print(f"\nПри ξ₁=ξ₂=ξ₃=ξ₄=ξ₅=0:")
print(f"η(x=0) = β₀ = {y0_point:.4f}")

# 2. ПАРАМЕТРИЧЕСКИЙ БУТСТРАП
B = 10000
np.random.seed(123)

beta0_boot = np.zeros(B)

for b in range(B):
    # Генерируем новые Y* из модели: Y* = Xβ̂ + ε*, ε* ~ N(0, σ̂²)
    Y_star = y_pred_original + np.random.normal(0, sigma_hat, n)
    
    # Оцениваем модель на бутстрап-данных
    model_boot = LinearRegression()
    model_boot.fit(X, Y_star)
    
    # Сохраняем β₀ (это прогноз в x=0)
    beta0_boot[b] = model_boot.intercept_

# 3. Доверительный интервал
alpha = 0.05
CI_lower = np.percentile(beta0_boot, 100 * alpha/2)
CI_upper = np.percentile(beta0_boot, 100 * (1 - alpha/2))

print(f"\nПАРАМЕТРИЧЕСКИЙ БУТСТРАП (B={B})")
print(f"Точечная оценка: η̂(0) = {y0_point:.4f}")
print(f"95% доверительный интервал: [{CI_lower:.4f}, {CI_upper:.4f}]")
print(f"Ширина интервала: {CI_upper - CI_lower:.4f}")

print(f"Бутстрап 95% CI:       [{CI_lower:.4f}, {CI_upper:.4f}]")

Модель: η = 1.9699 + (2.9485)·ξ1 + (-2.1777)·ξ2 + (0.5454)·ξ3 + (0.3370)·ξ4 + (-1.0800)·ξ5

При ξ₁=ξ₂=ξ₃=ξ₄=ξ₅=0:
η(x=0) = β₀ = 1.9699

ПАРАМЕТРИЧЕСКИЙ БУТСТРАП (B=10000)
----------------------------------------
Точечная оценка: η̂(0) = 1.9699
95% доверительный интервал: [1.5507, 2.3969]
Ширина интервала: 0.8462

Теоретический 95% CI:  [1.5385, 2.4013]
Бутстрап 95% CI:       [1.5507, 2.3969]


In [35]:
import json
import numpy as np

bootstrap_results = {
    "parametric_bootstrap": {
        "method": "Параметрический бутстрап",
        "B": int(B),
        "point_prediction": {
            "x": [0, 0, 0, 0, 0],
            "eta_hat": round(float(y0_point), 4)
        },
        "confidence_interval_95": {
            "lower": round(float(CI_lower), 4),
            "upper": round(float(CI_upper), 4),
            "width": round(float(CI_upper - CI_lower), 4)
        },
        "alpha": 0.05,
        "interpretation": f"С вероятностью 95% истинное значение η при ξ=0 лежит в интервале [{CI_lower:.4f}, {CI_upper:.4f}]"
    }
}

with open('1d.json', 'w', encoding='utf-8') as f:
    json.dump(bootstrap_results, f, indent=2, ensure_ascii=False)